Martin Ardila.


It is well known from class mechanics that by knowing the energy functional we can determine the trajectory of an object in phase space (momentum space) so we use for that the Lagrangian formalism and then it is canonically converted to the hamiltonian in terms of the generalized momentum of the system. Assuming at first that the energy is conserved (which in practical and real terms is not so, but serves as a first approach).

In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go

In this part i show how klepperian orbits are made in 2D, there are some visualization problems when the tryectories become parabolic or hyperbolic, this is fixed in later code for 3d development.

This code depends on an arbitrary l0 (which is the initial angular momentum starting at periapside)

In [2]:

# Constantes
G = 1.0
m1 = 1.0
m2 = 1.0
mu = m1 * m2 / (m1 + m2)

# Sistema Hamiltoniano en polares
def hamiltonian_polar(t, y):
    r, phi, pr, l = y

    drdt = pr / mu
    dphidt = l / (mu * r**2)
    dprdt = (l**2) / (mu * r**3) - (G * m1 * m2) / (r**2)
    dldt = 0

    return [drdt, dphidt, dprdt, dldt]

def compute_trajectory(r0, pr0, l0):
    y0 = [r0, 0.0, pr0, l0]
    t_span = (0, 50)
    t_eval = np.linspace(*t_span, 1000)
    sol = solve_ivp(hamiltonian_polar, t_span, y0, t_eval=t_eval, rtol=1e-9, atol=1e-9)
    r = sol.y[0]
    phi = sol.y[1]
    x = r * np.cos(phi)
    y = r * np.sin(phi)
    return x, y

r0_init = 1.0
pr0_init = 0.0
l0_values = np.linspace(0.5, 2.0, 5)

trajectories = [compute_trajectory(r0_init, pr0_init, l0) for l0 in l0_values]


# Crear figura base
fig = go.Figure()

# Añadir primer set de datos
x0, y0 = trajectories[0]
frame_data = [
    go.Scatter(
        x=x0[:1], y=y0[:1],
        mode='lines+markers',
        line=dict(color='blue'),
        marker=dict(color='red', size=6),
        name='Orbit'
    ),
    go.Scatter(
        x=[0], y=[0],
        mode='markers',
        marker=dict(color='orange', size=12),
        name='Center'
    )
]
fig.add_traces(frame_data)

# Crear frames para cada l0
frames = []
for idx, (x_traj, y_traj) in enumerate(trajectories):
    l0_val = l0_values[idx]
    frame_list = []
    for k in range(1, len(x_traj)):
        frame_list.append(go.Frame(
            data=[
                go.Scatter(
                    x=x_traj[:k+1], y=y_traj[:k+1],
                    mode='lines+markers',
                    line=dict(color='blue'),
                    marker=dict(color='red', size=6)
                ),
                go.Scatter(
                    x=[0], y=[0],
                    mode='markers',
                    marker=dict(color='orange', size=12)
                )
            ],
            name=f"l0={l0_val:.2f}-step{k}"
        ))
    frames += frame_list

fig.frames = frames

# Slider steps para cada l0
sliders = [
    {
        "steps": [
            {
                "method": "animate",
                "args": [
                    [f"l0={l0_val:.2f}-step{k}" for k in range(1, len(trajectories[idx][0]))],
                    {
                        "frame": {"duration": 20, "redraw": True},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ],
                "label": f"l0={l0_val:.2f}"
            } for idx, l0_val in enumerate(l0_values)
        ],
        "currentvalue": {"prefix": "Momento angular l0: "}
    }
]

# Layout
fig.update_layout(
    title="Two Body Problem — Hamiltonian Trajectory (Angular Momentum Slider)",
    xaxis_title="X",
    yaxis_title="Y",
    xaxis=dict(scaleanchor='y', scaleratio=1),
    sliders=sliders,
    updatemenus=[
        dict(
            type="buttons",
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[
                        None,
                        dict(frame=dict(duration=20, redraw=True), fromcurrent=True)
                    ]
                ),
                dict(
                    label="Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(frame=dict(duration=0, redraw=False), mode="immediate")
                    ]
                )
            ]
        )
    ]
)

fig.show()


Output hidden; open in https://colab.research.google.com to view.

OK uno tiene en los porblemas de orbitas que optimizar, L = T - Ueff , tal que Uff = U(r) + l^2 / (2*mu*r^2)

In [3]:
G = 1.0
m1 = 3.0
m2 = 1.5
mu = m1 * m2 / (m1 + m2)
gamma = G * m1 * m2


def hamiltonian_potential_effective(t, y, mu, gamma):
    r, phi, pr, l = y

    #eq canonicas
    #radial
    drdt = pr / mu
    dphidt = l / (mu * r**2)

    #de momento
    dprdt = (l**2) / (mu * r**3) - gamma / (r**2)

    # as always esta cosa siempre sale cte
    dldt = 0

    return [drdt, dphidt, dprdt, dldt]


def compute_trajectory(r0, pr0, l0, mu, gamma):
    y0 = [r0, 0.0, pr0, l0]
    t_span = (0, 50)
    t_eval = np.linspace(*t_span, 1500)

    sol = solve_ivp(hamiltonian_potential_effective, t_span, y0, t_eval=t_eval, args=(mu, gamma), rtol=1e-9, atol=1e-9)

    r = sol.y[0]
    phi = sol.y[1]
    x = r * np.cos(phi)
    y = r * np.sin(phi)
    z = np.zeros_like(x)

    return x, y, z, r

r0_init = 1.0
pr0_init = 0.0
l0_values = np.linspace(0.5, 3.0, 8)


trajectories = [compute_trajectory(r0_init, pr0_init, l0, mu, gamma) for l0 in l0_values]



In [ ]:

fig = go.Figure()


x0, y0, z0, r0_arr = trajectories[0]
fig.add_trace(go.Scatter3d(
    x=x0[:1], y=y0[:1], z=z0[:1],
    mode='lines+markers',
    line=dict(color='blue', width=4),
    marker=dict(color='red', size=4),
    name='Trayectoria'
))
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    marker=dict(color='orange', size=8),
    name='Centro de masas'
))

# Frames
frames = []
for idx, (x_traj, y_traj, z_traj, r_traj) in enumerate(trajectories):
    l0_val = l0_values[idx]
    for k in range(1, len(x_traj)):
        frames.append(go.Frame(
            data=[
                go.Scatter3d(
                    x=x_traj[:k+1], y=y_traj[:k+1], z=z_traj[:k+1],
                    mode='lines+markers',
                    line=dict(color='blue', width=4),
                    marker=dict(color='red', size=4)
                ),
                go.Scatter3d(
                    x=[0], y=[0], z=[0],
                    mode='markers',
                    marker=dict(color='orange', size=8)
                )
            ],
            name=f"l0={l0_val:.2f}-step{k}"
        ))

fig.frames = frames

# Slider
slider_steps = [
    {
        "method": "animate",
        "args": [
            [f"l0={l0_val:.2f}-step{k}" for k in range(1, len(trajectories[idx][0]))],
            {"frame": {"duration": 15, "redraw": True}, "mode": "immediate"}
        ],
        "label": f"{l0_val:.2f}"
    }
    for idx, l0_val in enumerate(l0_values)
]

# Layout
fig.update_layout(
    title=f"Hamiltonian con Potencial Efectivo | μ={mu:.2f}, γ={gamma:.2f}",
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    sliders=[{
        "steps": slider_steps,
        "currentvalue": {"prefix": "Momento angular l0: "}
    }],
    updatemenus=[
        dict(
            type="buttons",
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[
                        None,
                        {"frame": {"duration": 15, "redraw": True}, "fromcurrent": True}
                    ]
                ),
                dict(
                    label="Pause",
                    method="animate",
                    args=[
                        [None],
                        {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}
                    ]
                )
            ]
        )
    ]
)

fig.show()
